# Exercise 4 · Run both test suites headless and read the results

You will run the unit tests, then the browser tests without a window, and read what failed from the results.

**Goal:** Run the unit suite and the browser suite headless. Keep a trace of the failed test. Then read the results.

**Why it matters:** A headless test opens no window. A build server, a computer that runs the tests for every change, runs them headless. There, the saved results and the trace are all you can read.

**Before you start:**
- You did exercise 3 of today. Its Step 3 installed the browser for tests.

**What the check measures:** `unit.xml` shows 3 tests with no failure. `e2e.xml` shows 2 tests with 1 failure. A `trace.zip` is under `test-results/`. Your four answers are right.

**Done when:** the last cell prints `Done:`.

## The situation

`cases.py` holds the filter function and `cases.html` is the case list page. Each folder under `tests/` is a suite, a set of tests you run together. `tests/unit/` tests the function. `tests/e2e/` holds the end-to-end tests, which go through the browser. One browser test fails. You do not fix it here: you read the results. The files hold made-up data.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 3: the five files, `unit.xml`, `e2e.xml` and the failed test's `trace.zip`. It never overwrites a file.

Expect eight lines that start with `Wrote practice/`.

In [ ]:
FILES = {
    "cases.py": """\"\"\"Pick cases by status.\"\"\"


def filter_cases(cases: list[dict], status: str | None = None) -> list[dict]:
    \"\"\"Return the cases with a given status.

    Args:
        cases: The cases, each a dict with an "id" and a "status".
        status: The status to keep, or None to keep every case.

    Returns:
        The cases with that status, in their order.
    \"\"\"
    if status is None:
        return list(cases)
    return [case for case in cases if case["status"] == status]
""",
    "tests/unit/test_cases.py": """from cases import filter_cases

CASES = [{"id": "C-1", "status": "open"}, {"id": "C-2", "status": "open"}, {"id": "C-3", "status": "closed"}]


def test_open_returns_only_open_cases() -> None:
    assert [case["id"] for case in filter_cases(CASES, "open")] == ["C-1", "C-2"]


def test_no_status_returns_every_case() -> None:
    assert len(filter_cases(CASES)) == 3


def test_unknown_status_returns_no_cases() -> None:
    assert filter_cases(CASES, "pending") == []
""",
    "cases.html": """<!doctype html>
<html><head><meta charset="utf-8"><title>Case list</title></head>
<body>
<label for="status">Status</label>
<select id="status"><option value="">All</option><option value="Open">Open</option><option value="closed">Closed</option></select>
<table><tbody id="cases"></tbody></table>
<script>
const CASES = [{id: "C-1", status: "open"}, {id: "C-2", status: "open"}, {id: "C-3", status: "closed"}];
function show(status) {
  const body = document.getElementById("cases");
  clearTimeout(window.pending);
  body.innerHTML = "";
  window.pending = setTimeout(() => {
    for (const c of CASES.filter(c => !status || c.status === status)) {
      const row = document.createElement("tr");
      row.dataset.testid = "case-row";
      row.innerHTML = `<td>${c.id}</td><td>${c.status === "open" ? "Open" : "Closed"}</td>`;
      body.appendChild(row);
    }
  }, 300);
}
document.getElementById("status").addEventListener("change", e => show(e.target.value));
show("");
</script></body></html>
""",
    "tests/e2e/test_page.py": """from pathlib import Path

from playwright.sync_api import Page, expect

PAGE = Path(__file__).resolve().parents[2] / "cases.html"


def test_page_shows_every_case_at_first(page: Page) -> None:
    page.goto(PAGE.as_uri())
    expect(page.get_by_test_id("case-row")).to_have_count(3)


def test_filter_shows_only_open_cases(page: Page) -> None:
    page.goto(PAGE.as_uri())
    page.get_by_label("Status").select_option("Open")
    expect(page.get_by_test_id("case-row")).to_have_count(2)
""",
    "pytest.ini": "[pytest]\npythonpath = .\n",
}
# The results of Steps 2 and 3 (simulated, cut short from a real run)
FILES["unit.xml"] = """<?xml version="1.0" encoding="utf-8"?><testsuites name="pytest tests"><testsuite name="pytest" errors="0" failures="0" skipped="0" tests="3" time="0.017"><testcase classname="tests.unit.test_cases" name="test_open_returns_only_open_cases" time="0.000" /><testcase classname="tests.unit.test_cases" name="test_no_status_returns_every_case" time="0.000" /><testcase classname="tests.unit.test_cases" name="test_unknown_status_returns_no_cases" time="0.000" /></testsuite></testsuites>
"""
FILES["e2e.xml"] = """<?xml version="1.0" encoding="utf-8"?><testsuites name="pytest tests"><testsuite name="pytest" errors="0" failures="1" skipped="0" tests="2" time="6.332"><testcase classname="tests.e2e.test_page" name="test_page_shows_every_case_at_first[chromium]" time="1.185" /><testcase classname="tests.e2e.test_page" name="test_filter_shows_only_open_cases[chromium]" time="5.122"><failure message="AssertionError: Locator expected to have count '2'&#10;Actual value: 0">tests/e2e/test_page.py:16: AssertionError</failure></testcase></testsuite></testsuites>
"""

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

import zipfile

trace = PRACTICE / "test-results/tests-e2e-test-page-py-test-filter-shows-only-open-cases-chromium/trace.zip"
if trace.exists():
    print(f"Kept practice/{trace.relative_to(PRACTICE)}: it is already there.")
else:
    trace.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(trace, "w") as archive:
        archive.writestr("trace.trace", "simulated: a real trace.zip holds the recorded actions\n")
    print(f"Wrote practice/{trace.relative_to(PRACTICE)}")

## Step 2 · Run the unit suite first

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-04-run-headless-read-results/practice
pytest tests/unit --junitxml=unit.xml
```

Expect `3 passed` in less than a second. `--junitxml=unit.xml` saves the results in a file that build servers read. The unit suite is fast, so it runs first.

## Step 3 · Run the browser suite headless, with a trace

```sh
uv run --with pytest-playwright==0.10.0 pytest tests/e2e --tracing=retain-on-failure --junitxml=e2e.xml
```

Expect `1 failed, 1 passed` after about 6 seconds. No window opens. Read the lines under `FAILURES`.

### Learn: headless by default, and a trace for each failure

Playwright runs headless unless you ask for a window: "Run tests in headed mode (default: headless)" ([Running tests](https://playwright.dev/python/docs/test-runners)). A trace is a recording of each action, with the page before and after it. With `--tracing=retain-on-failure`, Playwright will "Record trace for each test, but remove all traces from successful test runs" ([Trace viewer](https://playwright.dev/python/docs/trace-viewer-intro)).

## Step 4 · Find the trace and open it

```sh
find test-results -name trace.zip
```

Expect one line: `test-results/tests-e2e-test-page-py-test-filter-shows-only-open-cases-chromium/trace.zip`. Only the failed test kept one. The path starts in `practice/`.

Open it in your browser. This works on your own computer and in the hosted workspace:

1. In the hosted workspace only: in the VS Code Explorer, right-click `trace.zip` and choose **Download**.
2. Open [trace.playwright.dev](https://trace.playwright.dev). Click **Select file** and choose `trace.zip`. "Trace Viewer loads the trace entirely in your browser and does not transmit any data externally."

On your own computer you can also run `uv run --with pytest-playwright==0.10.0 playwright show-trace` and the path. A Trace Viewer window opens.

Click each action in the list. See the page before and after it, and read its log.

## Step 5 · Answer from the results

Read the terminal, `e2e.xml` and the trace. Write your answers in the cell, then run it. Expect `Saved your answers.`

1. How many tests ran in each suite, and how many failed?
2. What is the name of the test that failed?
3. How many rows did that test expect, and how many did the page show?
4. In `e2e.xml`, find that test's `time`. How many seconds did it take, to the nearest second?

<details><summary>Show the answer</summary>

1. Unit: 3 ran, 0 failed. Browser: 2 ran, 1 failed.
2. `test_filter_shows_only_open_cases`. pytest adds `[chromium]`, the browser it ran in.
3. It expected 2 rows. The page showed 0: `Locator expected to have count '2'` and `Actual value: 0`.
4. About 5. Why: `expect` waited for its timeout, 5000 ms in the error's log, before it failed.

</details>

In [ ]:
UNIT_RAN, UNIT_FAILED = 3, 0  # question 1, the unit suite
BROWSER_RAN, BROWSER_FAILED = 2, 1  # question 1, the browser suite
FAILED_TEST = "test_filter_shows_only_open_cases"  # question 2
EXPECTED_ROWS, ACTUAL_ROWS = 2, 0  # question 3
FAILED_SECONDS = 5  # question 4
print("Saved your answers.")

## Check your work

Run the cell. Expect four `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: read unit.xml and e2e.xml, look for a trace.zip, and compare your answers
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")


def totals(name: str) -> tuple:
    root = ET.parse(PRACTICE / name).getroot()
    suites = [root] if root.tag == "testsuite" else root.findall("testsuite")
    return tuple(sum(int(s.get(key, 0)) for s in suites) for key in ("tests", "failures", "errors"))


def first_problem() -> str:
    for name, want, step in (("unit.xml", (3, 0, 0), "Step 2"), ("e2e.xml", (2, 1, 0), "Step 3")):
        if not (PRACTICE / name).is_file():
            return f"practice/{name} is missing. Run the command in {step} from practice/."
        got = totals(name)
        if got[2]:
            return f"{name} shows a test that stopped with an error. Is the browser installed (exercise 3, Step 3)?"
        if got != want:
            return f"{name} shows {got[0]} tests and {got[1]} failed, not {want[0]} and {want[1]}. Run {step} again."
        print(f"ok: {name} shows {got[0]} tests, {got[1]} failed")
    if not list(PRACTICE.glob("test-results/**/trace.zip")):
        return "no trace.zip under practice/test-results/. Run Step 3 with --tracing=retain-on-failure."
    print("ok: the failed test left a trace.zip")
    answers = [globals().get(name) for name in ("UNIT_RAN", "UNIT_FAILED", "BROWSER_RAN", "BROWSER_FAILED",
                                                "EXPECTED_ROWS", "ACTUAL_ROWS")]
    if [str(a).strip() for a in answers] != ["3", "0", "2", "1", "2", "0"]:
        return "a number in the Step 5 cell is missing or not right. Read Steps 2 and 3, then run that cell again."
    test = str(globals().get("FAILED_TEST", "")).split("::")[-1].split("[")[0].strip()
    if test != "test_filter_shows_only_open_cases":
        return "the failed test's name in the Step 5 cell is not right. Read the line under FAILURES (Step 3)."
    seconds = [float(case.get("time", 0)) for case in ET.parse(PRACTICE / "e2e.xml").getroot().iter("testcase")
               if case.find("failure") is not None]
    try:
        near = abs(float(str(globals().get("FAILED_SECONDS")).strip()) - seconds[0]) < 1
    except ValueError:
        near = False
    if not near:
        return "the seconds in the Step 5 cell are missing or not right. Read the failed test's time in e2e.xml."
    print("ok: your four answers are right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run Steps 2 and 3 again."
print(f"Not yet: {problem}" if problem else
      "Done: you ran both suites headless and read what failed from the results and the trace.")

## What you learned

- Run the fast unit suite first, then the browser suite. Both run headless, the same on your computer and a build server.
- `--junitxml` saves the results. `--tracing=retain-on-failure` keeps a trace of each failed test.

**Reflect:** A test fails on a build server. Which would you open first: the terminal output, the XML file or the trace? Why?